In [16]:
import os
import time
import urllib3
import requests
import pandas as pd

# Silencia o aviso de InsecureRequest (aquela mensagem amarela gigante)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [17]:
def buscar_concurso_api(numero_concurso):
    """Busca um concurso específico ou o último (se numero_concurso=0)."""    
    base_url = "https://servicebus2.caixa.gov.br/portaldeloterias/api/quina"
    url = f"{base_url}/{numero_concurso}" if numero_concurso > 0 else base_url
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/110.0.0.0 Safari/537.36",
        "Accept": "*/*",
        "Connection": "keep-alive"
    }
    
    try:
        # verify=False é usado pois o SSL da Caixa às vezes conflita com certas bibliotecas Python
        response = requests.get(url, headers=headers, verify=False, timeout=15)
        if response.status_code == 200:
            return response.json()
        else:
            print(f"Erro HTTP {response.status_code} no concurso {numero_concurso}")
    except Exception as e:
        print(f"Erro na requisição: {e}")
    return None

In [ ]:
def atualizar_dados():
    base_dir = os.getcwd()
    path_raw = os.path.join(base_dir, "..","data", "raw", "Quina.csv")
    
    # Verifica se o arquivo já existe e carrega dados atuais
    if os.path.exists(path_raw):
        df_atual = pd.read_csv(path_raw)
        ultimo_local = int(df_atual['numero'].max())
        print(f"Base local encontrada. Último concurso: {ultimo_local}")
    else:
        print(f"Criando nova base em: {path_raw}")
        df_atual = pd.DataFrame()
        ultimo_local = 0
        
    # Verificar o último na API
    info_ultimo = buscar_concurso_api(0)
    if not info_ultimo:
        print("Erro ao conectar com a API")
        return
    
    ultimo_api = info_ultimo['numero']
    print(f"Status: Local({ultimo_local}) | API({ultimo_api})")
    print(f"Faltam {ultimo_api} concursos.")

    if ultimo_local >= ultimo_api:
        print("Base já atualizada.")
        return
    
    # Loop de Captura com Checkpoints
    novos_sorteios = []
    contagem_check = 0
    
    try:
        for i in range(ultimo_local + 1, ultimo_api + 1):
            dados = buscar_concurso_api(i)
            if dados:
                novos_sorteios.append({
                    'numero': dados['numero'],
                    'data': dados['dataApuracao'],
                    'dezenas': ",".join(dados['listaDezenas'])
                })
                
                # Feedback visual
                if i % 10 == 0:
                    print(f"Progresso: {i}/{ultimo_api} ({(i/ultimo_api)*100:.2f}%)")
                    
                # Checkpoint: Salva a cada 100 novos registros para segurança
                contagem_check += 1
                if contagem_check >= 100:
                    df_checkpoint = pd.concat([df_atual, pd.DataFrame(novos_sorteios)], ignore_index=True)
                    df_checkpoint.to_csv(path_raw, sep=';', index=False)
                    contagem_check = 0
                    # Reduzimos o sleep levemente para acelerar, mas mantendo para evitar blocks
                    time.sleep(0.2)
            else:
                print(f"Falha no concurso {i}. Tentando próximo...")
                
    except KeyboardInterrupt:
        print("\nInterrompido pelo usuário. Salvando progresso atual...")
        
    # Salvamento final
    if novos_sorteios:
        df_final = pd.concat([df_atual, pd.DataFrame(novos_sorteios)], ignore_index=True)
        # Remove duplicatas por segurança (boa prática de engenharia de dados)
        df_final = df_final.drop_duplicates(subset=['numero'])
        df_final.to_csv(path_raw, sep=';', index=False)
        print(f"Carga finalizada! Total de registros: {len(df_final)}")
        
if __name__ == "__main__":
    atualizar_dados()

Base local encontrada. Último concurso: 50
Status: Local(50) | API(6982)
Faltam 6982 concursos.
Progresso? 60/6982 (0.86%)
Progresso? 70/6982 (1.00%)
Progresso? 80/6982 (1.15%)
Progresso? 90/6982 (1.29%)
Progresso? 100/6982 (1.43%)
Progresso? 110/6982 (1.58%)
Progresso? 120/6982 (1.72%)
Progresso? 130/6982 (1.86%)
Progresso? 140/6982 (2.01%)
Progresso? 150/6982 (2.15%)
Progresso? 160/6982 (2.29%)
Progresso? 170/6982 (2.43%)
Progresso? 180/6982 (2.58%)
Progresso? 190/6982 (2.72%)
Progresso? 200/6982 (2.86%)
Progresso? 210/6982 (3.01%)
Progresso? 220/6982 (3.15%)
Erro na requisição: HTTPSConnectionPool(host='servicebus2.caixa.gov.br', port=443): Read timed out. (read timeout=15)
Falha no concurso 228. Tentando próximo...
Progresso? 230/6982 (3.29%)
Progresso? 240/6982 (3.44%)
Progresso? 250/6982 (3.58%)
Progresso? 260/6982 (3.72%)
Progresso? 270/6982 (3.87%)
Progresso? 280/6982 (4.01%)
Progresso? 290/6982 (4.15%)
Progresso? 300/6982 (4.30%)
Progresso? 310/6982 (4.44%)
Erro na requisição: